In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.neighbors import BallTree

# ---- Config ----
DATASETS = Path("data/curated_datasets/minimal_example")
REGIONS = ["central-america", "australia-oceania", "south-america"]  # extend as more land
SECTORS = ["energy", "water", "transport", "telecom"]
THRESHOLDS_M = [300, 1_000, 5_000, 10_000, 50_000]

EARTH_R = 6_371_000  # meters


def load_assets_by_sector(region):
    """Returns dict of sector -> DataFrame[lat, lon, asset_type]."""
    out = {}
    for sector in SECTORS:
        manifest_path = next(
            DATASETS.glob(f"dataset_{region}_{sector}_*/manifest.json"), None
        )
        if manifest_path is None:
            continue
        with open(manifest_path) as f:
            manifest = json.load(f)
        records = manifest.get("records", manifest)
        if not records:
            continue
        out[sector] = pd.DataFrame([
            {"lat": r["lat"], "lon": r["lon"], "asset_type": r["asset_type"]}
            for r in records
        ])
    return out


def colocation_count(df_a, df_b, threshold_m):
    """Count rows in df_a that have at least one row in df_b within threshold_m."""
    if len(df_a) == 0 or len(df_b) == 0:
        return 0
    coords_a = np.radians(df_a[["lat", "lon"]].values)
    coords_b = np.radians(df_b[["lat", "lon"]].values)
    tree = BallTree(coords_b, metric="haversine")
    counts = tree.query_radius(coords_a, r=threshold_m / EARTH_R, count_only=True)
    return int((counts > 0).sum())


def compute_region(region):
    """Returns a long-format DataFrame: region, sector_a, sector_b, threshold_m, n_a, n_b, n_colocated, pct."""
    assets = load_assets_by_sector(region)
    print(f"\n=== {region} ===")
    for sector, df in assets.items():
        print(f"  {sector:10s}: {len(df):>6d} assets")

    rows = []
    for s_a in SECTORS:
        for s_b in SECTORS:
            if s_a == s_b:
                continue
            df_a = assets.get(s_a)
            df_b = assets.get(s_b)
            if df_a is None or df_b is None or len(df_a) == 0 or len(df_b) == 0:
                continue
            for thr in THRESHOLDS_M:
                n_colo = colocation_count(df_a, df_b, thr)
                rows.append({
                    "region":       region,
                    "sector_a":     s_a,
                    "sector_b":     s_b,
                    "threshold_m":  thr,
                    "n_a":          len(df_a),
                    "n_b":          len(df_b),
                    "n_colocated":  n_colo,
                    "pct":          100 * n_colo / len(df_a),
                })
    return pd.DataFrame(rows)


# ---- Run for all regions, combine ----
all_results = pd.concat([compute_region(r) for r in REGIONS], ignore_index=True)

# Save the long-format table
out_csv = Path("data/curated_datasets/minimal_example/_colocation_multiscale.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)
all_results.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

# ---- Pretty print: one matrix per (region, threshold) ----
for region in REGIONS:
    region_df = all_results[all_results["region"] == region]
    if region_df.empty:
        continue
    print(f"\n{'='*60}")
    print(f"Region: {region}")
    print(f"{'='*60}")
    for thr in THRESHOLDS_M:
        sub = region_df[region_df["threshold_m"] == thr]
        if sub.empty:
            continue
        print(f"\n  Threshold: {thr:>6,}m")
        # Pivot to sector_a (rows) × sector_b (cols)
        pivot = sub.pivot(index="sector_a", columns="sector_b", values="pct")
        # Format as percentages with 1 decimal
        print(pivot.to_string(float_format=lambda x: f"{x:>5.1f}%" if pd.notna(x) else "  ─  "))

# ---- Headline summary: average across regions, at each threshold, for each sector pair ----
print(f"\n{'='*60}")
print("Cross-region average co-location rate")
print(f"{'='*60}")
summary = (
    all_results
    .groupby(["sector_a", "sector_b", "threshold_m"])["pct"]
    .mean()
    .reset_index()
)
for thr in THRESHOLDS_M:
    sub = summary[summary["threshold_m"] == thr]
    if sub.empty:
        continue
    print(f"\n  Threshold: {thr:>6,}m (avg across {len(REGIONS)} regions)")
    pivot = sub.pivot(index="sector_a", columns="sector_b", values="pct")
    print(pivot.to_string(float_format=lambda x: f"{x:>5.1f}%" if pd.notna(x) else "  ─  "))


=== central-america ===
  energy    :    132 assets
  water     :     82 assets
  transport :     48 assets
  telecom   :      1 assets

=== australia-oceania ===
  energy    :     80 assets
  water     :     39 assets
  transport :     26 assets
  telecom   :      2 assets

=== south-america ===
  energy    :     81 assets
  water     :     41 assets
  transport :     33 assets
  telecom   :      4 assets

Saved: data\curated_datasets\minimal_example\_colocation_multiscale.csv

Region: central-america

  Threshold:    300m
sector_b   energy  telecom  transport  water
sector_a                                    
energy        NaN     0.0%       0.0%   1.5%
telecom      0.0%      NaN       0.0%   0.0%
transport    0.0%     0.0%        NaN   0.0%
water        2.4%     0.0%       0.0%    NaN

  Threshold:  1,000m
sector_b   energy  telecom  transport  water
sector_a                                    
energy        NaN     0.0%       2.3%   3.0%
telecom      0.0%      NaN       0.0%   0.

In [5]:
import json
from collections import Counter
from pathlib import Path

# The REAL datasets, not the minimal example
data_root = Path("data/curated_datasets/minimal_example")  # adjust path if different on your system
regions = ["central-america", "australia-oceania", "south-america", "africa"]

per_region_counts = {}
overall_counts = Counter()

for region in regions:
    region_counts = Counter()
    for sector in ["energy", "water", "transport", "telecom"]:
        # The full v1 dataset folder probably has a different naming convention
        # than the minimal_example. Adjust glob pattern as needed.
        manifest_path = next(
            data_root.glob(f"dataset_{region}_{sector}_minimal_v1*/manifest.json"), None
        )
        if not manifest_path:
            continue
        with open(manifest_path) as f:
            m = json.load(f)
        for r in m.get("records", []):
            region_counts[r["asset_type"]] += 1
            overall_counts[r["asset_type"]] += 1
    per_region_counts[region] = region_counts
    print(f"\n=== {region} ===")
    for asset_type, n in region_counts.most_common():
        print(f"  {asset_type:50s} {n:>5d}")

print(f"\n=== POOLED (all {len(regions)} regions) ===")
for asset_type, n in overall_counts.most_common():
    print(f"  {asset_type:50s} {n:>5d}")


=== central-america ===
  energy.generation.power_plant                         30
  water.storage_tank                                    30
  transport.airport                                     30
  energy.generation.solar_farm                          29
  water.wastewater.plant                                27
  energy.distribution.substation_untyped                26
  water.treatment.plant                                 25
  energy.distribution.substation                        16
  transport.train_station                               16
  energy.transmission.substation                        15
  energy.distribution.substation_minor                  12
  energy.generation.wind_farm                            4
  transport.port_terminal                                2
  telecom.data_center                                    1

=== australia-oceania ===
  energy.generation.solar_farm                          24
  energy.generation.power_plant                         23
  tr